In [ ]:
!pip -q install torch torchvision timm pandas scikit-learn Pillow

In [ ]:
import os, math, json, random
from dataclasses import dataclass
from typing import Tuple
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix, recall_score

print(torch.__version__, "CUDA available:", torch.cuda.is_available())

2.8.0+cu126 CUDA available: True


In [ ]:
@dataclass
class CFG:
    data_csv: str = "/content/drive/MyDrive/RARE25/Dataset/extracted/combined_image_labels.csv"  # <-- UPDATE THIS PATH
    out_dir: str = "/content/drive/MyDrive/RARE25/Dataset/newone/baseline_no_aug_combined_most"
    img_size: int = 224
    batch_size: int = 128
    num_workers: int = 4
    epochs: int = 10
    lr: float = 1e-3
    seed: int = 42
    model_name: str = "vit_base_patch16_224"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
print(CFG())

CFG(data_csv='/content/drive/MyDrive/RARE25/Dataset/extracted/combined_image_labels.csv', out_dir='/content/drive/MyDrive/RARE25/Dataset/newone/baseline_no_aug_combined_most', img_size=224, batch_size=128, num_workers=4, epochs=10, lr=0.001, seed=42, model_name='vit_base_patch16_224', device='cuda')


In [ ]:
import os, math, json, random
from dataclasses import dataclass
from typing import Tuple
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix, recall_score

print(torch.__version__, "CUDA available:", torch.cuda.is_available())

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def ensure_dir(d: str):
    os.makedirs(d, exist_ok=True)

def split_patientwise(df: pd.DataFrame, seed: int):
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=seed)
    train_idx, temp_idx = next(gss1.split(df, groups=df["patient_id"]))
    df_temp = df.iloc[temp_idx].reset_index(drop=True)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=1/3, random_state=seed+1)
    val_idx_rel, test_idx_rel = next(gss2.split(df_temp, groups=df_temp["patient_id"]))
    val_idx = df.index[temp_idx[val_idx_rel]]
    test_idx = df.index[temp_idx[test_idx_rel]]
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)

class BEImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"])

def apply_clahe(img: Image.Image):
    img_np = np.array(img)
    img_yuv = cv2.cvtColor(img_np, cv2.COLOR_RGB2YUV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    img_clahe = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
    return Image.fromarray(img_clahe)

def apply_hsv(img: Image.Image):
    img_np = np.array(img)
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
    # Example: Increase saturation (adjust value as needed)
    # You might want to add more sophisticated HSV adjustments here
    # img_hsv[:,:,1] = np.clip(img_hsv[:,:,1] * 1.2, 0, 255)
    img_hsv[:,:,1] = cv2.equalizeHist(img_hsv[:,:,1]) # Example: Equalize saturation
    img_hsv[:,:,2] = cv2.equalizeHist(img_hsv[:,:,2]) # Example: Equalize value
    img_hsv[:,:,0] = cv2.equalizeHist(img_hsv[:,:,0]) # Example: Equalize hue
    img_hsv[:,:,0] = img_hsv[:,:,0] # Example: Keep hue as is
    img_hsv[:,:,1] = np.clip(img_hsv[:,:,1], 0, 255)
    img_hsv[:,:,2] = np.clip(img_hsv[:,:,2], 0, 255)


    img_hsv_converted = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)
    return Image.fromarray(img_hsv_converted)


def get_transforms(img_size: int):
    return transforms.Compose([
        transforms.Lambda(lambda img: apply_clahe(img)),
        # transforms.Lambda(lambda img: apply_hsv(img)), # Removed HSV
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

2.8.0+cu126 CUDA available: True


In [ ]:
def calculate_ppv_at_recall(labels, probs, recall_threshold=0.9):
    """
    Calculates the Positive Predictive Value (PPV) at a specific recall threshold.

    Args:
        labels (np.ndarray): True labels (0 or 1).
        probs (np.ndarray): Predicted probabilities for the positive class.
        recall_threshold (float): The desired recall level.

    Returns:
        float: The PPV at the specified recall threshold, or None if recall_threshold cannot be reached.
    """
    # Sort predictions by probability in descending order
    sorted_indices = np.argsort(probs)[::-1]
    sorted_labels = labels[sorted_indices]

    tp = 0
    fp = 0
    total_positive = np.sum(labels)

    if total_positive == 0:
        return None # Cannot calculate recall if there are no positive labels

    for i in range(len(sorted_labels)):
        if sorted_labels[i] == 1:
            tp += 1
        else:
            fp += 1

        current_recall = tp / total_positive
        if current_recall >= recall_threshold:
            # Calculate PPV at this point
            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            return ppv
    return None # Recall threshold not reached

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self, model_name: str = "vit_base_patch16_224", num_classes: int = 2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        embed_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(128, num_classes)
        )
        # Freeze all params first
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Unfreeze only the LAST encoder block (paper-style)
        if hasattr(self.backbone, "blocks"):
            for p in self.backbone.blocks[-1].parameters():
                p.requires_grad = True

    def forward(self, x):
        feat = self.backbone(x)
        return self.classifier(feat)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs_all, labels_all = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:, 1]
        probs_all.append(probs.detach().cpu().numpy())
        labels_all.append(y.detach().cpu().numpy())
    probs = np.concatenate(probs_all)
    labels = np.concatenate(labels_all)
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    sens = recall_score(labels, preds, pos_label=1)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    roc_auc = None
    pr_auc = None
    if len(np.unique(labels)) == 2:
        try: roc_auc = roc_auc_score(labels, probs)
        except: pass
        try: pr_auc = average_precision_score(labels, probs)
        except: pass

    ppv_at_90recall = calculate_ppv_at_recall(labels, probs, recall_threshold=0.9)

    return {"acc":acc, "sensitivity":sens, "specificity":spec,
            "roc_auc": (None if roc_auc is None else float(roc_auc)),
            "pr_auc": (None if pr_auc is None else float(pr_auc)),
            "ppv_at_90recall": (None if ppv_at_90recall is None else float(ppv_at_90recall))}

def train_epoch(model, loader, device, optimizer, scaler, criterion):
    model.train()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    probs_all, labels_all = [], [] # Added for PPV@90Recall calculation

    for x, y in loader:
        x = x.to(device); y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device, enabled=(device=='cuda')):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)

        # Calculate accuracy
        _, preds = torch.max(logits, 1)
        correct_predictions += (preds == y).sum().item()
        total_samples += x.size(0)

        # Collect probabilities and labels for PPV@90Recall
        probs = torch.softmax(logits, dim=1)[:, 1]
        probs_all.append(probs.detach().cpu().numpy())
        labels_all.append(y.detach().cpu().numpy())


    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct_predictions / total_samples if total_samples > 0 else 0.0

    # Calculate PPV@90Recall for training data
    probs = np.concatenate(probs_all)
    labels = np.concatenate(labels_all)
    train_ppv_at_90recall = calculate_ppv_at_recall(labels, probs, recall_threshold=0.9)


    return avg_loss, accuracy, train_ppv_at_90recall # Return accuracy and PPV@90Recall

In [ ]:
set_seed(CFG.seed)
ensure_dir(CFG.out_dir)

# Load CSV
df = pd.read_csv(CFG.data_csv)
assert {'image_path','label','patient_id'}.issubset(df.columns), "CSV must contain filepath,label,patient_id"
df['image_path'] = df['image_path'].apply(os.path.abspath)

# Separate healthy and disease images
df_healthy = df[df['label'] == 0].reset_index(drop=True)
df_disease = df[df['label'] == 1].reset_index(drop=True)

# Sample 25 healthy images for validation
if len(df_healthy) > 25:
    healthy_patient_ids = df_healthy['patient_id'].unique()
    sampled_healthy_patient_ids = np.random.choice(healthy_patient_ids, size=min(5, len(healthy_patient_ids)), replace=False)
    df_val_healthy = df_healthy[df_healthy['patient_id'].isin(sampled_healthy_patient_ids)]
    df_train_healthy = df_healthy[~df_healthy['patient_id'].isin(sampled_healthy_patient_ids)]
else:
    df_val_healthy = df_healthy # Use all healthy images if less than 25
    df_train_healthy = pd.DataFrame(columns=df_healthy.columns) # Empty dataframe

# Sample 25 disease images for validation
if len(df_disease) > 25:
    disease_patient_ids = df_disease['patient_id'].unique()
    sampled_disease_patient_ids = np.random.choice(disease_patient_ids, size=min(5, len(disease_patient_ids)), replace=False)
    df_val_disease = df_disease[df_disease['patient_id'].isin(sampled_disease_patient_ids)]
    df_train_disease = df_disease[~df_disease['patient_id'].isin(sampled_disease_patient_ids)]
else:
    df_val_disease = df_disease # Use all disease images if less than 25
    df_train_disease = pd.DataFrame(columns=df_disease.columns) # Empty dataframe


# Combine validation sets
df_val = pd.concat([df_val_healthy, df_val_disease]).reset_index(drop=True)

# Combine remaining data for training
df_train = pd.concat([df_train_healthy, df_train_disease]).reset_index(drop=True)

# Augment label 1 data in training set
df_train_disease_augmented = pd.concat([df_train[df_train['label'] == 1]] * 3, ignore_index=True)
df_train = pd.concat([df_train[df_train['label'] == 0], df_train_disease_augmented]).reset_index(drop=True)


print("Patients (Train):", df_train['patient_id'].nunique())
print("Images (Train):", len(df_train))
print("Train class dist:", df_train['label'].value_counts().to_dict())

print("Patients (Validation):", df_val['patient_id'].nunique())
print("Images (Validation):", len(df_val))
print("Validation class dist:", df_val['label'].value_counts().to_dict())

# Datasets & loaders
print("Datasets Start")
tfm = get_transforms(CFG.img_size)
ds_train, ds_val = BEImageDataset(df_train, tfm), BEImageDataset(df_val, tfm)
dl_train = DataLoader(ds_train, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)


# Model/opt/loss
print("Datasets done")
model = ViTClassifier(model_name=CFG.model_name, num_classes=2).to(CFG.device)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CFG.lr)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(device=CFG.device, enabled=(CFG.device=='cuda'))
print("Model done")

best_score = -1.0
best_path = os.path.join(CFG.out_dir, "best.pth")
last_path = os.path.join(CFG.out_dir, "last.pth") # Define path for the last model
history = []

for epoch in range(1, CFG.epochs+1):
    print("Training")
    tr_loss, tr_acc, tr_ppv_at_90recall = train_epoch(model, dl_train, CFG.device, optimizer, scaler, criterion)
    val_metrics = evaluate(model, dl_val, CFG.device)
    # prefer PR-AUC if available, else accuracy
    score = val_metrics["pr_auc"] if val_metrics["pr_auc"] is not None else val_metrics["acc"]
    history.append({"epoch":epoch, "train_loss":tr_loss, "train_acc": tr_acc, "train_ppv_at_90recall": tr_ppv_at_90recall, **{f"val_{k}":v for k,v in val_metrics.items()}})


    # Format metrics for printing
    formatted_metrics = {k: f"{v:.4f}" if isinstance(v, (float, np.floating)) else v for k, v in val_metrics.items()}
    print(f"Epoch {epoch:02d} | train_loss {tr_loss:.4f} | train_acc {tr_acc:.4f} | train_ppv_at_90recall {tr_ppv_at_90recall:.4f} |", formatted_metrics)


    if score is not None and score > best_score:
        best_score = score
        torch.save({"model": model.state_dict()}, best_path)

# Save the last trained model after the loop
torch.save({"model": model.state_dict()}, last_path)


pd.DataFrame(history).to_csv(os.path.join(CFG.out_dir, "train_log.csv"), index=False)

# Test (load last) - Removed test set evaluation
# if os.path.exists(last_path):
#     ckpt = torch.load(last_path, map_location=CFG.device)
#     model.load_state_dict(ckpt["model"])

# test_metrics = evaluate(model, dl_test, CFG.device)
# print("TEST:", test_metrics)
# pd.DataFrame([test_metrics]).to_csv(os.path.join(CFG.out_dir, "test_metrics.csv"), index=False)
print("Saved to:", CFG.out_dir)

Patients (Train): 5866
Images (Train): 8272
Train class dist: {0: 4663, 1: 3609}
Patients (Validation): 10
Images (Validation): 10
Validation class dist: {0: 5, 1: 5}
Datasets Start
Datasets done
Model done
Training
Epoch 01 | train_loss 0.3842 | train_acc 0.8229 | train_ppv_at_90recall 0.7150 | {'acc': '0.8000', 'sensitivity': '0.8000', 'specificity': '0.8000', 'roc_auc': '0.9600', 'pr_auc': '0.9667', 'ppv_at_90recall': '0.8333'}
Training
Epoch 02 | train_loss 0.2250 | train_acc 0.9069 | train_ppv_at_90recall 0.8848 | {'acc': '0.8000', 'sensitivity': '0.6000', 'specificity': '1.0000', 'roc_auc': '1.0000', 'pr_auc': '1.0000', 'ppv_at_90recall': '1.0000'}
Training
Epoch 03 | train_loss 0.1803 | train_acc 0.9304 | train_ppv_at_90recall 0.9262 | {'acc': '0.6000', 'sensitivity': '0.4000', 'specificity': '0.8000', 'roc_auc': '0.8800', 'pr_auc': '0.8767', 'ppv_at_90recall': '0.8333'}
Training
Epoch 04 | train_loss 0.1365 | train_acc 0.9456 | train_ppv_at_90recall 0.9559 | {'acc': '0.8000', '